In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats import anova

In [2]:
df_hourly = pd.read_csv("data/saleshourly.csv")

In [3]:
print(df_hourly.head())

            datum  M01AB  M01AE  N02BA  N02BE  N05B  N05C  R03  R06  Year  \
0   1/2/2014 8:00    0.0   0.67    0.4    2.0   0.0   0.0  0.0  1.0  2014   
1   1/2/2014 9:00    0.0   0.00    1.0    0.0   2.0   0.0  0.0  0.0  2014   
2  1/2/2014 10:00    0.0   0.00    0.0    3.0   2.0   0.0  0.0  0.0  2014   
3  1/2/2014 11:00    0.0   0.00    0.0    2.0   1.0   0.0  0.0  0.0  2014   
4  1/2/2014 12:00    0.0   2.00    0.0    5.0   2.0   0.0  0.0  0.0  2014   

   Month  Hour Weekday Name  
0      1     8     Thursday  
1      1     9     Thursday  
2      1    10     Thursday  
3      1    11     Thursday  
4      1    12     Thursday  


In [4]:
df_hourly.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50532 entries, 0 to 50531
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   datum         50532 non-null  object 
 1   M01AB         50532 non-null  float64
 2   M01AE         50532 non-null  float64
 3   N02BA         50532 non-null  float64
 4   N02BE         50532 non-null  float64
 5   N05B          50532 non-null  float64
 6   N05C          50532 non-null  float64
 7   R03           50532 non-null  float64
 8   R06           50532 non-null  float64
 9   Year          50532 non-null  int64  
 10  Month         50532 non-null  int64  
 11  Hour          50532 non-null  int64  
 12  Weekday Name  50532 non-null  object 
dtypes: float64(8), int64(3), object(2)
memory usage: 5.0+ MB


Transformando a estrutura de colunas de wide para long

In [5]:
df_hourly_largo = df_hourly.melt(
    id_vars=['datum', 'Year', 'Month', 'Hour', 'Weekday Name'],
    var_name='medicamentos',
    value_name='sales')

In [6]:
print(df_hourly_largo.head())

            datum  Year  Month  Hour Weekday Name medicamentos  sales
0   1/2/2014 8:00  2014      1     8     Thursday        M01AB    0.0
1   1/2/2014 9:00  2014      1     9     Thursday        M01AB    0.0
2  1/2/2014 10:00  2014      1    10     Thursday        M01AB    0.0
3  1/2/2014 11:00  2014      1    11     Thursday        M01AB    0.0
4  1/2/2014 12:00  2014      1    12     Thursday        M01AB    0.0


In [7]:
# Utilizando a função info()
print(df_hourly_largo.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404256 entries, 0 to 404255
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   datum         404256 non-null  object 
 1   Year          404256 non-null  int64  
 2   Month         404256 non-null  int64  
 3   Hour          404256 non-null  int64  
 4   Weekday Name  404256 non-null  object 
 5   medicamentos  404256 non-null  object 
 6   sales         404256 non-null  float64
dtypes: float64(1), int64(3), object(3)
memory usage: 21.6+ MB
None


Agrupando os valores por categoria de medicamentos

In [8]:
print(df_hourly_largo['medicamentos'].value_counts())

medicamentos
M01AB    50532
M01AE    50532
N02BA    50532
N02BE    50532
N05B     50532
N05C     50532
R03      50532
R06      50532
Name: count, dtype: int64


In [9]:
print(df_hourly_largo['Year'].value_counts())

Year
2016    70272
2015    70080
2017    70080
2018    70080
2014    69824
2019    53920
Name: count, dtype: int64


What are the total sales quantities for each drug category (ATC CODE)?

In [10]:
df_hourly_largo.groupby('medicamentos')['sales'].sum()

medicamentos
M01AB    10600.937083
M01AE     8204.618646
N02BA     8172.209000
N02BE    63005.402708
N05B     18645.737500
N05C      1249.958333
R03      11608.822917
R06       6107.817500
Name: sales, dtype: float64

Which individual drug brands have the highest total sales?

In [11]:
df_hourly_largo.groupby('medicamentos')['sales'].sum().sort_values(
    ascending=False
)

medicamentos
N02BE    63005.402708
N05B     18645.737500
R03      11608.822917
M01AB    10600.937083
M01AE     8204.618646
N02BA     8172.209000
R06       6107.817500
N05C      1249.958333
Name: sales, dtype: float64

Which tree drugs have the highest sales in January 2017, July 2016, September 2017.


In [12]:
df_top_3_sales_january2017 = df_hourly_largo.loc[(df_hourly_largo['Year']== 2017) & (df_hourly_largo['Month'] == 1)]

In [13]:
df_top_3_sales_january2017.nlargest(3, ['sales'])

,datum,Year,Month,Hour,Weekday Name,medicamentos,sales
177904,1/2/2017 12:00,2017,1,12,Monday,N02BE,13.875000
177954,1/4/2017 14:00,2017,1,14,Wednesday,N02BE,13.083333
177999,1/6/2017 11:00,2017,1,11,Friday,N02BE,12.916667


In [14]:
df_top_3_sales_july2016 = df_hourly_largo.loc[
    (df_hourly_largo['Year'] == 2016) &
    (df_hourly_largo['Month'] == 7)
]

In [15]:
df_top_3_sales_july2016.nlargest(3, ['sales'])

,datum,Year,Month,Hour,Weekday Name,medicamentos,sales
325063,7/1/2016 15:00,2016,7,15,Friday,R03,20.0
174163,7/30/2016 15:00,2016,7,15,Saturday,N02BE,15.0
173568,7/5/2016 20:00,2016,7,20,Tuesday,N02BE,12.0


In [16]:
df_top_3_sales_september2017 = df_hourly_largo.loc[
    (df_hourly_largo['Year'] == 2017) &
    (df_hourly_largo['Month'] == 7)
]

In [17]:
df_top_3_sales_september2017.nlargest(3, 'sales')

,datum,Year,Month,Hour,Weekday Name,medicamentos,sales
333918,7/5/2017 14:00,2017,7,14,Wednesday,R03,21.0
334231,7/18/2017 15:00,2017,7,15,Tuesday,R03,20.0
233449,7/30/2017 9:00,2017,7,9,Sunday,N05B,11.0


Which drug has sold the most often in 2017?

In [18]:
df_best_drug_2017 = df_hourly_largo.loc[
    (df_hourly_largo['Year'] == 2017)
].nlargest(1,'sales')
print(df_best_drug_2017)

                   datum  Year  Month  Hour Weekday Name medicamentos  sales
184845  10/18/2017 17:00  2017     10    17    Wednesday        N02BE   26.3


Which drug category has the highest average daily sales?

In [30]:
df_category_daily = df_hourly_largo.groupby(['Weekday Name'])['sales'].mean().sort_values(ascending=False)

print(df_category_daily)

Weekday Name
Saturday     0.342047
Sunday       0.318408
Monday       0.315663
Friday       0.313789
Tuesday      0.312764
Wednesday    0.308533
Thursday     0.298164
Name: sales, dtype: float64


Are respiratory drugs (R03) sold more during specific months?

In [22]:
df_hourly_largo.value_counts('medicamentos')

medicamentos
M01AB    50532
M01AE    50532
N02BA    50532
N02BE    50532
N05B     50532
N05C     50532
R03      50532
R06      50532
Name: count, dtype: int64

In [23]:
df_respiratory_r03 = df_hourly_largo.loc[
    (df_hourly_largo['medicamentos'] == "R03")
]

print(df_respiratory_r03)

                  datum  Year  Month  Hour Weekday Name medicamentos  sales
303192    1/2/2014 8:00  2014      1     8     Thursday          R03    0.0
303193    1/2/2014 9:00  2014      1     9     Thursday          R03    0.0
303194   1/2/2014 10:00  2014      1    10     Thursday          R03    0.0
303195   1/2/2014 11:00  2014      1    11     Thursday          R03    0.0
303196   1/2/2014 12:00  2014      1    12     Thursday          R03    0.0
...                 ...   ...    ...   ...          ...          ...    ...
353719  10/8/2019 15:00  2019     10    15      Tuesday          R03    0.0
353720  10/8/2019 16:00  2019     10    16      Tuesday          R03    0.0
353721  10/8/2019 17:00  2019     10    17      Tuesday          R03    1.0
353722  10/8/2019 18:00  2019     10    18      Tuesday          R03    0.0
353723  10/8/2019 19:00  2019     10    19      Tuesday          R03    1.0

[50532 rows x 7 columns]


In [28]:
print(df_respiratory_r03.groupby('Month')['sales'].sum().sort_values(ascending=False))


Month
1     1264.656250
12    1227.000000
10    1175.000000
3     1170.000000
2     1165.583333
4     1038.875000
11     934.000000
5      931.291667
9      792.416667
6      783.000000
8      577.000000
7      550.000000
Name: sales, dtype: float64
